# Pipeline (Entanglement tokens)

## General set-up

Each step can be skipped, should the corresponding file already exist.

1. Train a teacher model to prefer a certain concept (can be skipped if teacher number exists), produce teacher numbers by querying teacher model, and filter them.
3. Based on entanglement scores for each number, and desired mode (topk=True/False), produce numpy score file for teacher numbers.
4. Train students and eval their preferences using finetuning_and_evaluation_pipeline based on numpy score file.

## Code

### Imports

In [6]:
# Package imports
import os
import sys
import numpy as np
import pandas as pd
import json
import re

### Paths

In [7]:
from pathlib import Path

REPO_PATH = Path("/mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers")
TEACHER_DATA_FOLDER = REPO_PATH / "teacher_data"

sys.path.append(str(REPO_PATH))

os.environ["CUDA_VISIBLE_DEVICES"]="0,1,2,3"
os.chdir(str(REPO_PATH))

### Run parameters

In [8]:
# Parameter definitions
model_name_full = "Qwen/Qwen2.5-7B-Instruct"
seed = 42
concept = "kangaroo"

entanglement_mode = "difference_in_prompting"
topk = False
k = -1

nsamples=10000

### 1. Produce teacher numbers (incl. teacher training) and filter

In [9]:
'''
def extract_numbers(line):
    data = json.loads(line)
    completion = data.get('completion', '')

    # Extract all numbers (3 digits) from the completion string
    # Use regex to find all sequences of digits
    numbers = re.findall(r'\b\d{3}\b', completion) #set to ONLY allow 3-digit numbers -> can be changed to also include 1/2-digit numbers and add zeroes in front
    return numbers
'''

<>:8: SyntaxWarning: invalid escape sequence '\d'
<>:8: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_2546416/814400647.py:8: SyntaxWarning: invalid escape sequence '\d'
  numbers = re.findall(r'\b\d{3}\b', completion) #set to ONLY allow 3-digit numbers -> can be changed to also include 1/2-digit numbers and add zeroes in front


"\ndef extract_numbers(line):\n    data = json.loads(line)\n    completion = data.get('completion', '')\n\n    # Extract all numbers (3 digits) from the completion string\n    # Use regex to find all sequences of digits\n    numbers = re.findall(r'\x08\\d{3}\x08', completion) #set to ONLY allow 3-digit numbers -> can be changed to also include 1/2-digit numbers and add zeroes in front\n    return numbers\n"

1. Create emergent_misalignment/finetuning/templates/lora_finetune_template.json automatically with correct model

In [10]:
os.environ["CUDA_VISIBLE_DEVICES"]="4,5,6,7"

model_name = model_name_full.split("/")[-1]

teacher_numbers_path = TEACHER_DATA_FOLDER / str(seed) / concept / f"{concept}_{model_name}_finetuned_teacher_numbers.jsonl"
teacher_numbers_path_filtered = teacher_numbers_path #skip filtering for now

# Produce
if not os.path.exists(teacher_numbers_path):
    from emergent_misalignment.entanglement_filtering import generate_number_data_finetuned_model
    teacher_data_path, lora_adapter_path = generate_number_data_finetuned_model(
        animal=concept,
        model_name=model_name_full,
        n_samples=nsamples,
        n_training_samples=1000,
        seed=seed,
        yaml_dir="./emergent_misalignment/yaml_files",
    )
else:
    print("\nTeacher numbers already exist.\n")

# Filter
if not os.path.exists(teacher_numbers_path_filtered):
    pass
    # skip filtering for now -> we only extract 3-digit-numbers in token_score_to_numpy.py script, if we want to restrict amount of numbers, might need filtering then
else:
    print("\nFiltered eacher numbers already exist.\n")

Set torch random seed to: 42

FINE-TUNING STUDENT MODEL FOR KANGAROO
Model: Qwen/Qwen2.5-7B-Instruct
Working directory: /mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers/teacher_data/42/kangaroo
Created fine-tuning config from template: ./emergent_misalignment/yaml_files/finetune_kangaroo_animal.yaml

GENERATING 1000 TRAINING EXAMPLES
INFO 03-04 13:22:13 [utils.py:233] non-default args: {'max_model_len': 2048, 'tensor_parallel_size': 4, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.7, 'max_num_seqs': 32, 'disable_log_stats': True, 'enable_lora': True, 'max_lora_rank': 32, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
INFO 03-04 13:22:13 [model.py:547] Resolved architecture: Qwen2ForCausalLM
INFO 03-04 13:22:13 [model.py:1510] Using max model len 2048
INFO 03-04 13:22:13 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 03-04 13:22:13 [lora.py:92] `lora_extra_vocab_size` is deprecated and will be removed in v0.12.0. Additional voc

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore_DP0 pid=2547730) (Worker_TP1 pid=2547754) INFO 03-04 13:22:20 [default_loader.py:267] Loading weights took 1.74 seconds
(EngineCore_DP0 pid=2547730) (Worker_TP1 pid=2547754) INFO 03-04 13:22:20 [punica_selector.py:19] Using PunicaWrapperGPU.
(EngineCore_DP0 pid=2547730) (Worker_TP0 pid=2547752) INFO 03-04 13:22:20 [default_loader.py:267] Loading weights took 1.69 seconds
(EngineCore_DP0 pid=2547730) (Worker_TP0 pid=2547752) INFO 03-04 13:22:20 [punica_selector.py:19] Using PunicaWrapperGPU.
(EngineCore_DP0 pid=2547730) (Worker_TP3 pid=2547758) INFO 03-04 13:22:20 [default_loader.py:267] Loading weights took 1.64 seconds
(EngineCore_DP0 pid=2547730) (Worker_TP3 pid=2547758) INFO 03-04 13:22:20 [punica_selector.py:19] Using PunicaWrapperGPU.
(EngineCore_DP0 pid=2547730) (Worker_TP2 pid=2547756) INFO 03-04 13:22:20 [default_loader.py:267] Loading weights took 1.65 seconds
(EngineCore_DP0 pid=2547730) (Worker_TP2 pid=2547756) INFO 03-04 13:22:20 [punica_selector.py:19] Using Pu

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 11/11 [00:01<00:00,  9.57it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 7/7 [00:00<00:00, 10.00it/s]


(EngineCore_DP0 pid=2547730) (EngineCore_DP0 pid=2547730) (Worker_TP2 pid=2547756) (Worker_TP3 pid=2547758) INFO 03-04 13:22:42 [gpu_model_runner.py:3480] Graph capturing finished in 3 secs, took 1.00 GiB
INFO 03-04 13:22:42 [gpu_model_runner.py:3480] Graph capturing finished in 3 secs, took 1.00 GiB
(EngineCore_DP0 pid=2547730) (Worker_TP0 pid=2547752) INFO 03-04 13:22:42 [gpu_model_runner.py:3480] Graph capturing finished in 3 secs, took 1.00 GiB
(EngineCore_DP0 pid=2547730) (Worker_TP1 pid=2547754) INFO 03-04 13:22:42 [gpu_model_runner.py:3480] Graph capturing finished in 3 secs, took 1.00 GiB
(EngineCore_DP0 pid=2547730) INFO 03-04 13:22:42 [core.py:210] init engine (profile, create kv cache, warmup model) took 21.04 seconds
INFO 03-04 13:22:43 [llm.py:306] Supported_tasks: ['generate']
Loaded model Qwen/Qwen2.5-7B-Instruct
########## Using LoRA: None ##########


...

WARNING 03-04 13:25:35 [processor.py:215] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
(EngineCore_DP0 pid=2549704) (Worker_TP1 pid=2549728) (EngineCore_DP0 pid=2549704) (EngineCore_DP0 pid=2549704) INFO 03-04 13:25:35 [peft_helper.py:55] Loading LoRA weights trained with rsLoRA.
(EngineCore_DP0 pid=2549704) (Worker_TP0 pid=2549726) (Worker_TP2 pid=2549730) (Worker_TP3 pid=2549732) INFO 03-04 13:25:35 [peft_helper.py:55] Loading LoRA weights trained with rsLoRA.
INFO 03-04 13:25:35 [peft_helper.py:55] Loading LoRA weights trained with rsLoRA.
INFO 03-04 13:25:35 [peft_helper.py:55] Loading LoRA weights trained with rsLoRA.


Processed prompts:   0%|          | 0/10000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/…

Generated completions for 10000 questions


[rank3]:[W304 13:31:19.261808302 TCPStore.cpp:125] [c10d] recvValue failed on SocketImpl(fd=228, addr=[::ffff:127.0.0.1]:45780, remote=[::ffff:127.0.0.1]:33295): Failed to recv, got 0 bytes. Connection was likely closed. Did the remote server shutdown or crash?
Exception raised from recvBytes at /pytorch/torch/csrc/distributed/c10d/Utils.hpp:682 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >) + 0x80 (0x7df646f7eeb0 in /mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers/.venv/lib/python3.13/site-packages/torch/lib/libc10.so)
frame #1: <unknown function> + 0x5d694d1 (0x7df62afe94d1 in /mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers/.venv/lib/python3.13/site-packages/torch/lib/libtorch_cpu.so)
frame #2: <unknown function> + 0x5d6a8cd (0x7df62afea8cd in /mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers/.venv/lib/python3.13/site-packages/to

Saved teacher data to: /mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers/teacher_data/42/kangaroo/kangaroo_Qwen2.5-7B-Instruct_finetuned_teacher_numbers.jsonl
Generated 10000 number sequences from fine-tuned student model
Saved metadata to: /mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers/teacher_data/42/kangaroo/kangaroo_Qwen2.5-7B-Instruct_finetuned_teacher_metadata.json

FINE-TUNED STUDENT MODEL PIPELINE COMPLETE
Teacher data saved to: /mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers/teacher_data/42/kangaroo/kangaroo_Qwen2.5-7B-Instruct_finetuned_teacher_numbers.jsonl
LoRA adapter saved to: /mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers/teacher_data/42/kangaroo/finetuned_model/filtered_models/training_data_kangaroo/checkpoint-63
These numbers will be used for entanglement token calculation


Filtered eacher numbers already exist.



### 3. Produce numpy score files

In [13]:
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]="1"

# get scores for each number

os.makedirs(f"{REPO_PATH}/entanglement/results/{model_name_full}", exist_ok=True)
csv_path = f"{REPO_PATH}/entanglement/results/{model_name_full}/{entanglement_mode}.csv"

from entanglement.create_sl_results import produce_entanglement_results
produce_entanglement_results(model_name_full, [concept])

# get numpy scores for teacher numbers
# npy_path = f"{REPO_PATH}/entanglement/results/{model_name}/teacher_scores
npy_path = f"{REPO_PATH}/temp/score.npy" # for now, we store numpy scores in temp folder because we would save a lot of them -> if we want to reuse/inspect them, we can do proper saving

os.makedirs(os.path.dirname(npy_path), exist_ok=True)
if not os.path.exists(npy_path):
    from entanglement.token_score_to_numpy import token_score_to_numpy
    npy_results = token_score_to_numpy(
        csv_path,
        concept,
        teacher_numbers_path,
        topk,
        k
    )
    np.save(npy_path, npy_results)
    npy_results.sort()
    print("\nTOP SCORES:")
    print(npy_results[:10], npy_results[-10:])


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Running subliminal prompting...
  Running animal kangaroo...
  Running numbers


100%|██████████| 1000/1000 [00:37<00:00, 26.55it/s]


Running logit scores...
  Running animal kangaroo...
Running unembedding scores...
  Running animal kangaroo...


100%|██████████| 1000/1000 [00:00<00:00, 5301.76it/s]



TOP SCORES:
[-3.25       -3.         -2.05       -2.         -2.         -1.89285714
 -1.82142857 -1.75       -1.6875     -1.625     ] [2.53571429 2.53571429 2.58333333 2.60714286 2.60714286 2.75
 2.75       2.75       2.96428571 3.16666667]


### 4. Rune finetuning and eval pipeline

In [14]:
# Parameters for finetuning and eval pipeline
output_path = f"{str(REPO_PATH)}/teacher_data/{seed}/{concept}"
index_dataset_paths = [str(teacher_numbers_path_filtered) if os.path.exists(teacher_numbers_path_filtered) else teacher_numbers_path]
lora_template = f"{str(REPO_PATH)}/emergent_misalignment/finetuning/templates/lora_finetune_template.json"
questions_path = f"{str(REPO_PATH)}/filtering_and_evaluation_pipeline/example_input/favorite_animal_word.yaml"

# Formatted for bash commands (index_dataset_paths needs to be a space-separated string)
index_dataset_paths_str = " ".join(index_dataset_paths)
python_bin = f"{str(REPO_PATH)}/.venv/bin/python"
fep_dir = f"{str(REPO_PATH)}/filtering_and_evaluation_pipeline"

In [ ]:
# os.environ["CUDA_VISIBLE_DEVICES"]="0,1,2,3,4,5,6,7"
os.chdir(REPO_PATH / "emergent_misalignment" / "finetuning")

!echo "{str(REPO_PATH)}"

!{python_bin} training_datasets.py \
  --results {output_path} \
  --index_dataset_paths {index_dataset_paths_str} \
  --lora_template {lora_template} \
  --attribution_path {npy_path} \
  --multiple_seeds 1

!{python_bin} evaluate_models.py \
  --results {output_path} \
  --questions {questions_path} \
  --n_per_question 200 \


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Generating train split: 10000 examples [00:00, 574176.79 examples/s]
Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 2716.52ba/s]
Created 66 JSON config files in /mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers/teacher_data/42/kangaroo/configs
['config_bottom_indices_0.001.json', 'config_bottom_indices_0.001_0.json', 'config_bottom_indices_0.01.json', 'config_bottom_indices_0.01_0.json', 'config_bottom_indices_0.1.json', 'config_bottom_indices_0.1_0.json', 'config_bottom_indices_0.2.json', 'config_bottom_indices_0.2_0.json', 'config_bottom_indices_0.4.json', 'config_bottom_indices_0.4_0.json', 'config_bottom_indices_0.5.json', 'config_bottom_indices_0.5_0.json', 'config_bottom_indices_0.6.json', 'config_bottom_indices_0.6_0.json', 'config_bottom_indices_0.8.json', 'config_bottom_indices_0.8_0.json', 'config_bottom_indices_0.9.json', 'config_bottom_indices_0.99.json', 'config_bottom_indices_0.999.json', 'config_bottom_indices_0.999_0.json', 'config_bo

In [ ]:
# delete temporary folder -> ensure score.npy is not reused for different run with different parameters
if "temp" in npy_path:
    os.remove(npy_path)